# Etapa 1 — Aprendizado Simbólico (Árvore de Decisão)

**Pergunta:** *o município da Amazônia Legal tem presença de facção criminosa em 2024, dado o seu porte e o padrão de violência registrado nele entre 2021 e 2023?*

- **Rótulo:** *Cartografias da Violência na Amazônia*, 3ª edição (Instituto Mãe Crioula / FBSP), Tabelas 13–21.
- **Atributos:** SINESP VDE / MJSP, 2021–2023, por município, e população do IBGE (Censo 2022).
- **Unidade:** 808 municípios dos 9 estados da Amazônia Legal.

> Observação: "sem registro" no Cartografias significa que os pesquisadores **não identificaram** facção, não que ela certamente não exista. A população vem do IBGE (Censo 2022) e define o **porte**; as taxas são por 100 mil habitantes.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from utils import montar_tabela, atributos_numericos_log, ATRIBUTOS_CAT, DESCRICAO, SEED
np.random.seed(SEED)
pd.set_option("display.width", 140)
d = montar_tabela()
print(d.shape)

(808, 115)


In [2]:
pd.DataFrame({"descrição": DESCRICAO}).loc[ATRIBUTOS_CAT + ["ALVO", "REGIME"]]

,descrição
PORTE,"População (Censo 2022): 0=pequeno (<10 mil), 1..."
TXH,Taxa anual de homicídio doloso 2021–23 por 100...
TXT,Taxa anual de tentativas de homicídio 2021–23 ...
ESC,Teve morte a esclarecer em 2021–23 (0/1)
LATRO,Teve latrocínio em 2021–23 (0/1)
UF,Unidade da Federação (9 estados da Amazônia Le...
ALVO,Presença de facção em 2024 segundo o Cartograf...
REGIME,"2024: 0=sem registro, 1=uma facção (domínio), ..."


In [3]:
print(d["ALVO"].value_counts().rename({0: "sem registro de facção", 1: "com facção"}))
print(f"\nProporção com facção: {d['ALVO'].mean():.3f}  |  'sempre não' acerta {1 - d['ALVO'].mean():.1%}")
print("\nPor UF:"); print(d.groupby("UF")["ALVO"].agg(["size", "mean"]).round(2).sort_values("mean", ascending=False))

ALVO
sem registro de facção    549
com facção                259
Name: count, dtype: int64

Proporção com facção: 0.321  |  'sempre não' acerta 67.9%

Por UF:
    size  mean
UF            
AC    22  1.00
RR    15  0.93
PA   144  0.51
RO    52  0.50
AM    62  0.32
AP    16  0.31
MT   141  0.30
MA   217  0.22
TO   139  0.06


## 1. Entropia e ganho de informação

$$H(S) = -\sum_c p_c \log_2 p_c \qquad Ganho(S, A) = H(S) - \sum_v \frac{|S_v|}{|S|} H(S_v)$$

In [4]:
def entropia(y):
    y = np.asarray(y)
    if len(y) == 0:
        return 0.0
    p = np.bincount(y) / len(y)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def ganho(df, atributo, alvo="ALVO"):
    H_pai = entropia(df[alvo])
    H_filhos = sum(len(sub) / len(df) * entropia(sub[alvo]) for _, sub in df.groupby(atributo))
    return H_pai - H_filhos

print(f"H(ALVO) na raiz = {entropia(d['ALVO']):.4f} bits")
pd.Series({a: ganho(d, a) for a in ATRIBUTOS_CAT}).sort_values(ascending=False).to_frame("ganho na raiz").round(4)

H(ALVO) na raiz = 0.9050 bits


,ganho na raiz
PORTE,0.1654
UF,0.1509
LATRO,0.0696
ESC,0.0304
TXH,0.0267
TXT,0.0167


## 2. Cálculo dos 3 primeiros níveis (para o relatório)

In [5]:
def detalhar(df, atributos, nivel=1, max_nivel=3, caminho="raiz"):
    if nivel > max_nivel or len(atributos) == 0 or df["ALVO"].nunique() == 1:
        return
    gs = {a: ganho(df, a) for a in atributos}
    melhor = max(gs, key=gs.get)
    print(f"{'  '*(nivel-1)}[nível {nivel}] {caminho} | n={len(df)} | H={entropia(df['ALVO']):.4f} "
          f"| melhor={melhor} (ganho={gs[melhor]:.4f})")
    for v, sub in df.groupby(melhor):
        detalhar(sub, [a for a in atributos if a != melhor], nivel + 1, max_nivel, f"{caminho} > {melhor}={v}")

detalhar(d, ATRIBUTOS_CAT)

[nível 1] raiz | n=808 | H=0.9050 | melhor=PORTE (ganho=0.1654)
  [nível 2] raiz > PORTE=0 | n=267 | H=0.4725 | melhor=UF (ganho=0.1668)
    [nível 3] raiz > PORTE=0 > UF=AM | n=2 | H=1.0000 | melhor=TXH (ganho=1.0000)
    [nível 3] raiz > PORTE=0 > UF=MA | n=42 | H=0.1623 | melhor=LATRO (ganho=0.0695)
    [nível 3] raiz > PORTE=0 > UF=MT | n=67 | H=0.5692 | melhor=TXT (ganho=0.0221)
    [nível 3] raiz > PORTE=0 > UF=PA | n=12 | H=0.4138 | melhor=TXT (ganho=0.1434)
    [nível 3] raiz > PORTE=0 > UF=RO | n=22 | H=0.9760 | melhor=TXT (ganho=0.2979)
    [nível 3] raiz > PORTE=0 > UF=TO | n=111 | H=0.0741 | melhor=TXT (ganho=0.0191)
  [nível 2] raiz > PORTE=1 | n=435 | H=0.9183 | melhor=UF (ganho=0.1331)
    [nível 3] raiz > PORTE=1 > UF=AM | n=48 | H=0.6962 | melhor=TXH (ganho=0.1068)
    [nível 3] raiz > PORTE=1 > UF=AP | n=8 | H=0.9544 | melhor=TXH (ganho=0.2044)
    [nível 3] raiz > PORTE=1 > UF=MA | n=153 | H=0.7399 | melhor=LATRO (ganho=0.0400)
    [nível 3] raiz > PORTE=1 > UF=MT | 

    [nível 3] raiz > PORTE=2 > UF=RO | n=7 | H=0.8631 | melhor=TXH (ganho=0.4696)
    [nível 3] raiz > PORTE=2 > UF=TO | n=5 | H=0.7219 | melhor=TXT (ganho=0.7219)


## 3. LEARN-DECISION-TREE implementada do zero

In [6]:
class ArvoreID3:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth

    def fit(self, X, y):
        df = X.copy(); df["ALVO"] = np.asarray(y)
        self.raiz = self._construir(df, list(X.columns), 0, int(df["ALVO"].mode()[0]))
        return self

    def _construir(self, df, atributos, prof, maj_pai):
        if len(df) == 0:
            return {"folha": maj_pai}
        maj = int(df["ALVO"].mode()[0])
        if df["ALVO"].nunique() == 1 or not atributos or (self.max_depth is not None and prof >= self.max_depth):
            return {"folha": maj, "n": len(df), "p1": float(df["ALVO"].mean())}
        gs = {a: ganho(df, a) for a in atributos}
        A = max(gs, key=gs.get)
        if gs[A] <= 1e-12:
            return {"folha": maj, "n": len(df), "p1": float(df["ALVO"].mean())}
        filhos = {v: self._construir(sub, [a for a in atributos if a != A], prof + 1, maj)
                  for v, sub in df.groupby(A)}
        return {"atributo": A, "filhos": filhos, "maj": maj}

    def _prever1(self, x, no):
        while "folha" not in no:
            v = x[no["atributo"]]
            if v not in no["filhos"]:
                return no["maj"]
            no = no["filhos"][v]
        return no["folha"]

    def predict(self, X):
        return np.array([self._prever1(r, self.raiz) for _, r in X.iterrows()])

    def imprimir(self, no=None, ind=""):
        no = no or self.raiz
        if "folha" in no:
            print(f"{ind}-> ALVO={no['folha']}  (n={no.get('n','?')}, P(facção)={no.get('p1', float('nan')):.2f})")
            return
        for v, f in no["filhos"].items():
            print(f"{ind}{no['atributo']} = {v}")
            self.imprimir(f, ind + "    ")

X = d[ATRIBUTOS_CAT]; y = d["ALVO"].values
arv = ArvoreID3(max_depth=2).fit(X, y)
arv.imprimir()

PORTE = 0
    UF = AC
        -> ALVO=1  (n=3, P(facção)=1.00)
    UF = AM
        -> ALVO=0  (n=2, P(facção)=0.50)
    UF = AP
        -> ALVO=0  (n=6, P(facção)=0.00)
    UF = MA
        -> ALVO=0  (n=42, P(facção)=0.02)
    UF = MT
        -> ALVO=0  (n=67, P(facção)=0.13)
    UF = PA
        -> ALVO=0  (n=12, P(facção)=0.08)
    UF = RO
        -> ALVO=0  (n=22, P(facção)=0.41)
    UF = RR
        -> ALVO=1  (n=2, P(facção)=1.00)
    UF = TO
        -> ALVO=0  (n=111, P(facção)=0.01)
PORTE = 1
    UF = AC
        -> ALVO=1  (n=17, P(facção)=1.00)
    UF = AM
        -> ALVO=0  (n=48, P(facção)=0.19)
    UF = AP
        -> ALVO=0  (n=8, P(facção)=0.38)
    UF = MA
        -> ALVO=0  (n=153, P(facção)=0.21)
    UF = MT
        -> ALVO=0  (n=61, P(facção)=0.34)
    UF = PA
        -> ALVO=0  (n=90, P(facção)=0.40)
    UF = RO
        -> ALVO=1  (n=23, P(facção)=0.52)
    UF = RR
        -> ALVO=1  (n=12, P(facção)=0.92)
    UF = TO
        -> ALVO=0  (n=23, P(facção)=0.17)
PORTE = 2
 

## 4. Validação cruzada 5-fold: implementação própria × scikit-learn

In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.tree import DecisionTreeClassifier
from utils import tercis_do_treino

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
X_sk = pd.get_dummies(X, columns=["UF"]).astype(int)

def avaliar(fazer_modelo, Xm, nome):
    linhas = []
    for tr, te in cv.split(Xm, y):
        Xf = tercis_do_treino(d, Xm, tr)      # cortes de TXH e TXT só com o treino
        m = fazer_modelo().fit(Xf.iloc[tr], y[tr])
        p_tr, p_te = m.predict(Xf.iloc[tr]), m.predict(Xf.iloc[te])
        linhas.append([accuracy_score(y[tr], p_tr), accuracy_score(y[te], p_te),
                       balanced_accuracy_score(y[te], p_te), f1_score(y[te], p_te, zero_division=0)])
    m = np.mean(linhas, axis=0)
    return pd.Series(m, index=["acurácia treino", "acurácia teste", "acurácia balanceada", "F1 (classe facção)"], name=nome)

res = pd.DataFrame([
    avaliar(lambda: ArvoreID3(max_depth=None), X, "ID3 própria, sem poda"),
    avaliar(lambda: ArvoreID3(max_depth=3), X, "ID3 própria, prof. 3"),
    avaliar(lambda: DecisionTreeClassifier(criterion="entropy", random_state=SEED), X_sk, "sklearn, sem poda"),
    avaliar(lambda: DecisionTreeClassifier(criterion="entropy", max_depth=4, random_state=SEED), X_sk, "sklearn, prof. 4"),
    avaliar(lambda: DecisionTreeClassifier(criterion="entropy", max_depth=4, class_weight="balanced", random_state=SEED), X_sk, "sklearn, prof. 4, balanceada"),
]).round(3)
res

,acurácia treino,acurácia teste,acurácia balanceada,F1 (classe facção)
"ID3 própria, sem poda",0.874,0.749,0.687,0.566
"ID3 própria, prof. 3",0.824,0.786,0.708,0.590
"sklearn, sem poda",0.875,0.745,0.683,0.560
"sklearn, prof. 4",0.799,0.782,0.679,0.532
"sklearn, prof. 4, balanceada",0.756,0.735,0.685,0.560


In [8]:
profs = range(1, 9)
acc_te, bal_te = [], []
for p in profs:
    s = avaliar(lambda: DecisionTreeClassifier(criterion="entropy", max_depth=p, random_state=SEED), X_sk, p)
    acc_te.append(s["acurácia teste"]); bal_te.append(s["acurácia balanceada"])
plt.figure(figsize=(6, 3.5))
plt.plot(profs, acc_te, "o-", label="acurácia (teste)")
plt.plot(profs, bal_te, "s-", label="acurácia balanceada (teste)")
plt.xlabel("profundidade máxima"); plt.ylabel("métrica"); plt.grid(alpha=.3); plt.legend()
plt.title("Escolha da profundidade (5-fold)")
plt.tight_layout(); plt.savefig("figuras/etapa1_profundidade.png", dpi=150)
print("Melhor profundidade (acurácia):", list(profs)[int(np.argmax(acc_te))])

Melhor profundidade (acurácia): 5


## 5. Limitações da lógica: monotonicidade

Pegamos a regra mais simples e "segura" que os dados sugerem e acrescentamos um fato novo.

In [9]:
regra = d["PORTE"] == 0
print("Regra R: PORTE = pequeno (menos de 10 mil habitantes)  =>  sem facção")
print(f"Municípios cobertos: {regra.sum()}  |  acerto da regra: {1 - d.loc[regra, 'ALVO'].mean():.1%}")
print("\nAgora acrescentamos UM fato novo: a UF do município.")
t = d[regra].groupby("UF")["ALVO"].agg(municipios="size", com_faccao="sum", proporcao="mean").round(2)
print(t.sort_values("proporcao", ascending=False))

Regra R: PORTE = pequeno (menos de 10 mil habitantes)  =>  sem facção
Municípios cobertos: 267  |  acerto da regra: 89.9%

Agora acrescentamos UM fato novo: a UF do município.
    municipios  com_faccao  proporcao
UF                                   
AC           3           3       1.00
RR           2           2       1.00
AM           2           1       0.50
RO          22           9       0.41
MT          67           9       0.13
PA          12           1       0.08
MA          42           1       0.02
TO         111           1       0.01
AP           6           0       0.00


**Discussão.** A regra "município pequeno ⇒ sem facção" cobre 267 municípios e acerta em 89,9% deles. Mas, quando acrescentamos o fato "o município fica no Acre" ou "em Roraima", a conclusão se inverte: todos os municípios pequenos desses estados têm facção (3 de 3 e 2 de 2). Em Rondônia, 9 de 22 também têm. São estados de fronteira, na rota da cocaína: lá a facção chega até às cidades pequenas.

Na lógica clássica, o raciocínio é **monotônico**: se ∀x (Pequeno(x) → ¬Facção(x)) foi aceita, nenhum fato novo a derruba. No mundo real, um fato novo invalida a conclusão anterior. Isso mostra o limite das regras rígidas e motiva os graus de crença das etapas seguintes.